# 08 — Synthèse Finale
**Toutes les RQ** : Agrégation de tous les résultats expérimentaux.

Produit les tableaux et graphiques finaux pour le mémoire.

In [ ]:
import sys, os, pandas as pd, numpy as np, glob, json
sys.path.insert(0, os.path.abspath(".."))

from experiments.registry import ExperimentLog
from notebooks.lib.plotter import barplot, heatmap, boxplot, histogram
from notebooks.lib.reporter import to_csv, to_excel, to_json

# Charger tous les résultats JSON
all_results = {}
for fpath in sorted(glob.glob("../outputs/json/*.json")):
    name = os.path.basename(fpath).replace("_", " ").replace(".json", "")
    with open(fpath) as f:
        all_results[name] = json.load(f)

print(f"{len(all_results)} fichiers de résultats chargés")
for k in all_results:
    n = len(all_results[k].get('results', []))
    print(f"  {k}: {n} enregistrements")

In [ ]:
# Tableau récapitulatif : meilleure configuration pour chaque métrique
def best_config(results, metric):
    """Trouve la config qui maximise une métrique."""
    best = None
    best_score = -1
    for name, data in results.items():
        scores = [r.get(metric, 0) for r in data.get('results', [])]
        if not scores:
            continue
        avg = np.mean(scores)
        if avg > best_score:
            best_score = avg
            best = (name, avg, data.get('parameters', {}))
    return best

metrics = ['faithfulness', 'answer_relevancy',
           'contextual_precision', 'contextual_recall']

print("=== Meilleure configuration par métrique ===")
rows = []
for m in metrics:
    bc = best_config(all_results, m)
    if bc:
        rows.append({'Métrique': m, 'Configuration': bc[0],
                     'Score moyen': round(bc[1], 4)})
        print(f"  {m}: {bc[0]} (score={bc[1]:.4f})")

summary = pd.DataFrame(rows)
to_excel(rows, 'tableau_recapitulatif')

In [ ]:
# Graphique récapitulatif comparatif
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, metric in zip(axes.flatten(), metrics):
    configs = []
    scores = []
    for name, data in all_results.items():
        s = [r.get(metric, 0) for r in data.get('results', [])]
        if s:
            configs.append(name[:25])
            scores.append(np.mean(s))

    colors = ['#1F384B' if s == max(scores) else '#A0A0A0' for s in scores]
    ax.barh(configs, scores, color=colors)
    ax.set_title(f"{metric}", fontweight='bold')
    ax.set_xlim(0, 1)

plt.tight_layout()
plt.savefig('../outputs/figures/comparisons/synthese_finale.png', dpi=200, bbox_inches='tight')
plt.show()

In [ ]:
# Export final pour le mémoire
final = {
    "date_generation": pd.Timestamp.now().isoformat(),
    "nombre_experiences": len(all_results),
    "benchmark": "30 questions (4 niveaux x 6 catégories)",
    "meilleurs_resultats": rows,
    "details": {k: v['results'] for k, v in all_results.items()},
}

to_json(final, 'resultats_memoire_final')
to_excel(rows, 'resultats_memoire_tableau')
print("Résultats finaux exportés dans outputs/json/ et outputs/excel/")
print("Figures dans outputs/figures/comparisons/")